In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
import os
if not os.path.exists('/content/cdt-alzheimer-screening'):
    !git clone https://github.com/wiambenadder/cdt-alzheimer-screening.git
%cd cdt-alzheimer-screening
!git pull 2>/dev/null
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, '/content/cdt-alzheimer-screening')
print("setup ok")

Mounted at /content/drive
/content
Cloning into 'cdt-alzheimer-screening'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 64 (delta 21), reused 31 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 621.04 KiB | 10.89 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/cdt-alzheimer-screening
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 153.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.9 MB/s eta 0:00:00
setup ok


In [ ]:
%%writefile /content/cdt-alzheimer-screening/src/data.py
"""
NHATS CDT dataset: loading, participant-disjoint splits, this is for class imbalance
handling.

Rubric items:
  - #1  Train/val/test split with the following ratios (70/15/15)
  - #3  DataLoader batching + shuffling
  - #8  Normalization of input
  - #9  Basic preprocessing (resize, format conversion)
  - #10 Preprocessing pipeline addressing >=2 data quality challenges
        Challenge 1: Class imbalance (weighted sampling + class weights)
        Challenge 2: Image quality / format heterogeneity (TIFF -> RGB, resize)
        Challenge 3: Participant leakage across longitudinal rounds (disjoint split)
"""
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight

from .config import (
    DataConfig, SEED, NUM_CLASSES, LABELS_CSV, RAW_IMAGES_DIR,
)


class CDTDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, int(row["label"])


def load_labels(
    labels_csv: Path = LABELS_CSV,
    images_dir: Path = RAW_IMAGES_DIR,
    image_ext: str = ".tif",
    score_column: str = "cdt_score",
    id_column: str = "participant_id",
) -> pd.DataFrame:
    df = pd.read_csv(labels_csv)
    df = df.rename(columns={score_column: "label", id_column: "participant_id"})
    if "image_path" not in df.columns:
        df["image_path"] = df["participant_id"].apply(
            lambda pid: str(images_dir / f"{pid}{image_ext}")
        )
    df = df[df["label"].between(0, NUM_CLASSES - 1)].copy()
    df["label"] = df["label"].astype(int)
    exists_mask = df["image_path"].apply(lambda p: Path(p).exists())
    dropped = (~exists_mask).sum()
    if dropped > 0:
        print(f"[data] dropped {dropped} rows with missing image files")
    df = df[exists_mask].reset_index(drop=True)
    print(f"[data] loaded {len(df)} labeled clock images")
    return df


def stratified_split(df: pd.DataFrame, cfg: DataConfig):
    """Legacy random-stratified split. Kept for backward compatibility."""
    assert abs(cfg.train_ratio + cfg.val_ratio + cfg.test_ratio - 1.0) < 1e-6
    train_df, temp_df = train_test_split(
        df, test_size=1 - cfg.train_ratio, stratify=df["label"], random_state=SEED,
    )
    val_size_within_temp = cfg.val_ratio / (cfg.val_ratio + cfg.test_ratio)
    val_df, test_df = train_test_split(
        temp_df, train_size=val_size_within_temp,
        stratify=temp_df["label"], random_state=SEED,
    )
    print(f"[split] train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


def participant_disjoint_split(df: pd.DataFrame, cfg: DataConfig):
    """
    Leak-safe split: we inforce every participant to appear in EXACTLY ONE of train/val/test.

    NHATS is longitudinal, so the same SPID draws clocks in multiple rounds.
    A random-stratified split would put the same person's Round-5 clock in
    training and their Round-11 clock in test, which will inflate the held-out accuracy.
    This split prevents that.

    Stratification on label is approximate here: GroupShuffleSplit splits
    on groups (participant_id), then we verify class distribution is
    reasonable across the three splits.
    """
    rng = np.random.default_rng(SEED)

    # Step 1: train vs val+test
    gss1 = GroupShuffleSplit(
        n_splits=1, train_size=cfg.train_ratio, random_state=SEED,
    )
    train_idx, temp_idx = next(gss1.split(df, groups=df["participant_id"]))
    train_df = df.iloc[train_idx].reset_index(drop=True)
    temp_df = df.iloc[temp_idx].reset_index(drop=True)

    # Step 2: val vs test within the temp set
    val_ratio_in_temp = cfg.val_ratio / (cfg.val_ratio + cfg.test_ratio)
    gss2 = GroupShuffleSplit(
        n_splits=1, train_size=val_ratio_in_temp, random_state=SEED,
    )
    val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["participant_id"]))
    val_df = temp_df.iloc[val_idx].reset_index(drop=True)
    test_df = temp_df.iloc[test_idx].reset_index(drop=True)

    # we make sure that there is no participant overlap
    train_ids = set(train_df["participant_id"])
    val_ids   = set(val_df["participant_id"])
    test_ids  = set(test_df["participant_id"])
    assert not (train_ids & val_ids), "Participant leak: train/val overlap"
    assert not (train_ids & test_ids), "Participant leak: train/test overlap"
    assert not (val_ids   & test_ids), "Participant leak: val/test overlap"

    print(f"[split] participant-disjoint split:")
    print(f"  train: {len(train_df):,} clocks from {len(train_ids):,} participants")
    print(f"  val:   {len(val_df):,} clocks from {len(val_ids):,} participants")
    print(f"  test:  {len(test_df):,} clocks from {len(test_ids):,} participants")
    print(f"\n  train class dist:\n{train_df['label'].value_counts().sort_index().to_string()}")
    print(f"\n  test class dist:\n{test_df['label'].value_counts().sort_index().to_string()}")
    return train_df, val_df, test_df


def compute_class_weights(train_df: pd.DataFrame) -> torch.Tensor:
    labels = train_df["label"].values
    classes_present = np.unique(labels)
    weights_present = compute_class_weight(
        class_weight="balanced", classes=classes_present, y=labels,
    )
    full = np.ones(NUM_CLASSES, dtype=np.float32)
    for c, w in zip(classes_present, weights_present):
        full[c] = w
    return torch.tensor(full, dtype=torch.float32)


def make_weighted_sampler(train_df: pd.DataFrame) -> WeightedRandomSampler:
    class_counts = train_df["label"].value_counts().sort_index()
    sample_weights = train_df["label"].apply(lambda y: 1.0 / class_counts[y]).values
    return WeightedRandomSampler(
        weights=sample_weights, num_samples=len(sample_weights), replacement=True,
    )


def build_dataloaders(train_ds, val_ds, test_ds, cfg: DataConfig, sampler=None):
    shuffle_train = sampler is None
    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=shuffle_train,
        sampler=sampler, num_workers=cfg.num_workers, pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=True,
    )
    return train_loader, val_loader, test_loader

Overwriting /content/cdt-alzheimer-screening/src/data.py


In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

# we check the path is accessible
from pathlib import Path
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/cdt-data/nhats_raw')
print(f"Drive data dir exists: {DRIVE_DATA_DIR.exists()}")
print(f"Round folders: {sorted([p.name for p in DRIVE_DATA_DIR.iterdir() if p.is_dir()])}")

Mounted at /content/drive
Drive data dir exists: True
Round folders: ['round_01', 'round_02', 'round_03', 'round_04', 'round_05', 'round_06', 'round_07', 'round_08', 'round_09', 'round_10', 'round_11', 'round_12', 'round_13', 'round_14']


In [ ]:
import shutil, time
from pathlib import Path
import pandas as pd

LOCAL_DATA_DIR = Path('/content/nhats_local')
LOCAL_DATA_DIR.mkdir(exist_ok=True)
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/cdt-data/nhats_raw')

t0 = time.time()
for rf in sorted(DRIVE_DATA_DIR.iterdir()):
    if not rf.is_dir(): continue
    dst = LOCAL_DATA_DIR / rf.name
    if dst.exists() and len(list(dst.glob('*.tif'))) > 0:
        print(f"[skip] {rf.name}")
        continue
    print(f"[copy] {rf.name}...")
    shutil.copytree(rf, dst, dirs_exist_ok=True)

n = sum(1 for _ in LOCAL_DATA_DIR.rglob('*.tif'))
print(f"\n[done] {n:,} TIFFs staged in {(time.time()-t0)/60:.1f} min")

# we rebuild the local labels
drive_labels = pd.read_csv('/content/drive/MyDrive/cdt-data/labels.csv')
drive_labels['image_path'] = drive_labels['image_path'].str.replace(
    '/content/drive/MyDrive/cdt-data/nhats_raw/',
    '/content/nhats_local/', regex=False,
)
LOCAL_LABELS = Path('/content/cdt_labels_local.csv')
drive_labels.to_csv(LOCAL_LABELS, index=False)
print(f"[ok] local labels -> {LOCAL_LABELS}")

[copy] round_01...
[copy] round_02...
[copy] round_03...
[copy] round_04...
[copy] round_05...
[copy] round_06...
[copy] round_07...
[copy] round_08...
[copy] round_09...
[copy] round_10...
[copy] round_11...
[copy] round_12...
[copy] round_13...
[copy] round_14...

[done] 73,769 TIFFs staged in 49.1 min
[ok] local labels -> /content/cdt_labels_local.csv


In [ ]:
import importlib, src.config, src.data, src.augmentation, src.train, src.models, src.evaluate
for m in [src.config, src.data, src.augmentation, src.train, src.models, src.evaluate]:
    importlib.reload(m)

import src.config as cfg
from pathlib import Path

cfg.LABELS_CSV  = Path('/content/cdt_labels_local.csv')
cfg.MODELS_DIR  = Path('/content/drive/MyDrive/cdt-data/models')
cfg.RESULTS_DIR = Path('/content/drive/MyDrive/cdt-data/results')
cfg.MODELS_DIR.mkdir(parents=True, exist_ok=True)
cfg.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
from src.data import participant_disjoint_split, compute_class_weights, CDTDataset, build_dataloaders
from src.config import DataConfig, TrainConfig, AugConfig
from src.augmentation import build_train_transform, build_eval_transform

df = pd.read_csv(cfg.LABELS_CSV)
df = df.rename(columns={'cdt_score': 'label'})[['participant_id', 'label', 'image_path']]
df['participant_id'] = df['participant_id'].astype(str)
print(f"Total labels: {len(df):,}")

data_cfg = DataConfig(batch_size=32, num_workers=2)
train_df, val_df, test_df = participant_disjoint_split(df, data_cfg)
class_weights = compute_class_weights(train_df)
print(f"\nClass weights: {class_weights.tolist()}")

Total labels: 59,417
[split] participant-disjoint split:
  train: 41,551 clocks from 9,549 participants
  val:   8,928 clocks from 2,046 participants
  test:  8,938 clocks from 2,047 participants

  train class dist:
label
0      359
1     1352
2     4674
3     8904
4    15427
5    10835

  test class dist:
label
0      94
1     323
2     996
3    1976
4    3187
5    2362

Class weights: [19.290157318115234, 5.122164726257324, 1.4816360473632812, 0.7777590751647949, 0.44889912009239197, 0.6391478180885315]


In [ ]:
aug_cfg  = AugConfig()
train_tf = build_train_transform(data_cfg.image_size, aug_cfg, use_aug=True)
eval_tf  = build_eval_transform(data_cfg.image_size)

train_ds = CDTDataset(train_df, transform=train_tf)
val_ds   = CDTDataset(val_df,   transform=eval_tf)
test_ds  = CDTDataset(test_df,  transform=eval_tf)

train_loader, val_loader, test_loader = build_dataloaders(
    train_ds, val_ds, test_ds, data_cfg,
)
print(f"batches/epoch train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}")

batches/epoch train=1298, val=279, test=280


In [ ]:
import torch, json, time
from src.train import train as train_fn
from src.evaluate import collect_predictions, compute_all_metrics
from src.models import get_model
from src.utils import get_device, load_checkpoint

device = get_device()
print(f"device: {device}")

RESULTS = {}   # this accumulates as we train

def run_experiment(model_name: str, run_name: str, epochs: int = 15,
                   freeze_backbone: bool = False,
                   lr_head: float = 1e-3, lr_backbone: float = 1e-5,
                   weight_decay: float = 1e-4, dropout: float = 0.3):
    """Train one config, evaluate on test, store metrics in RESULTS."""
    train_cfg = TrainConfig(
        epochs=epochs, lr_head=lr_head, lr_backbone=lr_backbone,
        weight_decay=weight_decay, dropout=dropout,
        early_stopping_patience=4, mixed_precision=True,
        lr_scheduler='cosine', optimizer='adamw',
    )
    print(f"\n{'='*70}\nRUN: {run_name}  ({model_name}, freeze={freeze_backbone}, epochs={epochs})\n{'='*70}")
    t0 = time.time()

    summary = train_fn(
        model_name=model_name,
        train_loader=train_loader, val_loader=val_loader,
        class_weights=class_weights, cfg=train_cfg,
        run_name=run_name, freeze_backbone=freeze_backbone,
    )

    # we evaluate the best checkpoint on the test set
    model, _, _ = get_model(model_name, freeze_backbone=freeze_backbone, dropout=dropout)
    model = load_checkpoint(model, summary['checkpoint_path'], device=device).to(device)
    y_true, y_pred, y_probs = collect_predictions(model, test_loader, device)
    metrics = compute_all_metrics(y_true, y_pred, y_probs)
    metrics['run_name'] = run_name
    metrics['best_epoch'] = summary['best_epoch']
    metrics['minutes'] = (time.time() - t0) / 60
    RESULTS[run_name] = metrics

    # we persist after every run for this not to crash
    with open(cfg.RESULTS_DIR / 'finetuning_results.json', 'w') as f:
        json.dump(RESULTS, f, indent=2)

    print(f"\n[{run_name}] DONE in {metrics['minutes']:.1f} min")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k:20s} {v:.4f}")
    del model
    torch.cuda.empty_cache()
    return metrics

device: cuda


In [ ]:
# VGG16 fine-tuned, took 5-6 hours
run_experiment('vgg16', 'vgg16_ft', epochs=15)

# EfficientNet-B0 fine-tuned, this was done later
run_experiment('efficientnet_b0', 'effb0_ft', epochs=15)

# ViT-B/16 fine-tuned,this was done later
run_experiment('vit_b16', 'vit_b16_ft', epochs=15)


RUN: vgg16_ft  (vgg16, freeze=False, epochs=15)
[train] run=vgg16_ft device=cuda
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 231MB/s]
/content/cdt-alzheimer-screening/src/train.py:155: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(cfg.mixed_precision and device == "cuda"))


[train] params total=134,285,126 trainable=134,285,126


epoch 0 [train]:   0%|          | 0/1298 [00:00<?, ?it/s]/content/cdt-alzheimer-screening/src/train.py:75: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.mixed_precision):
/content/cdt-alzheimer-screening/src/train.py:87: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[train] ep=00 tr_loss=1.5493 tr_acc=0.426 val_loss=1.4975 val_acc=0.525


...

[train] ep=04 tr_loss=1.1023 tr_acc=0.577 val_loss=1.2611 val_acc=0.624


[train] ep=05 tr_loss=1.0771 tr_acc=0.589 val_loss=1.1287 val_acc=0.616


[train] ep=06 tr_loss=1.0367 tr_acc=0.603 val_loss=1.0431 val_acc=0.621


[train] ep=07 tr_loss=0.9881 tr_acc=0.614 val_loss=1.0609 val_acc=0.627


KeyboardInterrupt: 

In [14]:
# Find where the checkpoint actually is
from pathlib import Path

# Possible locations
candidates = [
    '/content/drive/MyDrive/cdt-data/models/',
    '/content/cdt-alzheimer-screening/models/',
    '/content/drive/MyDrive/cdt-data/',
    '/content/',
]

for c in candidates:
    p = Path(c)
    if p.exists():
        pt_files = list(p.rglob('*.pt'))
        print(f"\n{c}:")
        for f in pt_files:
            size_mb = f.stat().st_size / 1e6
            print(f"  {f}  ({size_mb:.0f} MB)")
    else:
        print(f"\n{c}: NOT FOUND")

# Also check what cfg.MODELS_DIR is right now
import src.config as cfg_mod
print(f"\nCurrent cfg.MODELS_DIR: {cfg_mod.MODELS_DIR}")


/content/drive/MyDrive/cdt-data/models/:
  /content/drive/MyDrive/cdt-data/models/vgg16_frozen_baseline_best.pt  (537 MB)

/content/cdt-alzheimer-screening/models/:
  /content/cdt-alzheimer-screening/models/vgg16_ft_best.pt  (537 MB)

/content/drive/MyDrive/cdt-data/:
  /content/drive/MyDrive/cdt-data/models/vgg16_frozen_baseline_best.pt  (537 MB)

/content/:
  /content/drive/MyDrive/cdt-data/models/vgg16_frozen_baseline_best.pt  (537 MB)
  /content/cdt-alzheimer-screening/models/vgg16_ft_best.pt  (537 MB)

Current cfg.MODELS_DIR: /content/drive/MyDrive/cdt-data/models


In [15]:
!ls -la /content/cdt-alzheimer-screening/models/

total 524580
drwxr-xr-x 2 root root      4096 Apr 24 15:19 .
drwxr-xr-x 9 root root      4096 Apr 24 13:35 ..
-rw-r--r-- 1 root root       774 Apr 24 13:35 README.md
-rw-r--r-- 1 root root 537152807 Apr 24 20:32 vgg16_ft_best.pt


In [16]:
import torch, json, shutil, importlib
from pathlib import Path
from src.models import get_model
from src.evaluate import collect_predictions, compute_all_metrics
from src.utils import load_checkpoint, get_device

# 1. we copy VGG16 checkpoint to Drive for safety
local_ckpt = Path('/content/cdt-alzheimer-screening/models/vgg16_ft_best.pt')
drive_ckpt = Path('/content/drive/MyDrive/cdt-data/models/vgg16_ft_best.pt')
drive_ckpt.parent.mkdir(parents=True, exist_ok=True)
if not drive_ckpt.exists():
    print(f"Copying to Drive ({local_ckpt.stat().st_size/1e6:.0f} MB)...")
    shutil.copy(local_ckpt, drive_ckpt)
    print(f"[ok] copied to {drive_ckpt}")

# 2. we evaluate on the test set
device = get_device()
model, _, _ = get_model('vgg16', freeze_backbone=False, dropout=0.3)
model = load_checkpoint(model, str(local_ckpt), device=device).to(device)

y_true, y_pred, y_probs = collect_predictions(model, test_loader, device)
metrics = compute_all_metrics(y_true, y_pred, y_probs)
metrics['run_name']   = 'vgg16_ft'
metrics['best_epoch'] = 6
metrics['note']       = 'trained 8 epochs, stopped manually for time budget'
RESULTS['vgg16_ft']   = metrics

with open(cfg.RESULTS_DIR / 'finetuning_results.json', 'w') as f:
    json.dump(RESULTS, f, indent=2)

print("\n=== VGG16 fine-tuned — FINAL TEST METRICS ===")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:20s} {v:.4f}")
    else:
        print(f"  {k:20s} {v}")

del model
torch.cuda.empty_cache()

# 3. we fix the MODELS_DIR so EfficientNet and ViT save to Drive
import src.config, src.train, src.utils
src.config.MODELS_DIR = Path('/content/drive/MyDrive/cdt-data/models')
src.train.MODELS_DIR  = Path('/content/drive/MyDrive/cdt-data/models')
# we also patch save_checkpoint to double-save to Drive as belt-and-suspenders
_orig_save = src.utils.save_checkpoint
def save_checkpoint_robust(model, path):
    path = Path(path)
    # we force path into Drive if it's on local disk
    if '/content/cdt-alzheimer-screening/models' in str(path):
        path = Path('/content/drive/MyDrive/cdt-data/models') / path.name
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), path)
src.utils.save_checkpoint = save_checkpoint_robust
src.train.save_checkpoint = save_checkpoint_robust

print(f"\n[ok] MODELS_DIR is now: {src.config.MODELS_DIR}")
print(f"[ok] Next training runs (EfficientNet, ViT) will save directly to Drive")

Copying to Drive (537 MB)...
[ok] copied to /content/drive/MyDrive/cdt-data/models/vgg16_ft_best.pt

=== VGG16 fine-tuned — FINAL TEST METRICS ===
  accuracy             0.6160
  macro_f1             0.5842
  weighted_f1          0.6120
  macro_precision      0.5766
  macro_recall         0.6043
  quadratic_kappa      0.7813
  macro_auc            0.9026
  run_name             vgg16_ft
  best_epoch           6
  note                 trained 8 epochs, stopped manually for time budget

[ok] MODELS_DIR is now: /content/drive/MyDrive/cdt-data/models
[ok] Next training runs (EfficientNet, ViT) will save directly to Drive


In [18]:
import importlib
from torch.utils.data import DataLoader

# we rebuild the dataloaders with aggressive parallel config
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=6,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
    num_workers=6,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)
test_loader = DataLoader(
    test_ds,
    batch_size=32,
    shuffle=False,
    num_workers=6,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)
print(f"[ok] dataloaders rebuilt with 6 workers + persistent + prefetch=4")
print(f"train batches: {len(train_loader)}")
print(f"val   batches: {len(val_loader)}")
print(f"test  batches: {len(test_loader)}")

[ok] dataloaders rebuilt with 6 workers + persistent + prefetch=4
train batches: 1298
val   batches: 279
test  batches: 280


In [19]:
import time
import torch

t0 = time.time()
n_batches_to_test = 30

device = 'cuda'
for i, (imgs, labels) in enumerate(train_loader):
    imgs = imgs.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)
    if i >= n_batches_to_test - 1:
        break

elapsed = time.time() - t0
sec_per_batch = elapsed / n_batches_to_test
projected_epoch = sec_per_batch * 1298 / 60   # we make it 1298 batches/epoch
print(f"Tested {n_batches_to_test} batches in {elapsed:.1f}s")
print(f"Seconds per batch: {sec_per_batch:.2f}")
print(f"Projected epoch time: {projected_epoch:.1f} min")

Tested 30 batches in 21.8s
Seconds per batch: 0.73
Projected epoch time: 15.7 min


In [20]:
# EfficientNet-B0 fine-tuned
# the number of epochs was chosen based on the speed test
run_experiment('efficientnet_b0', 'effb0_ft', epochs=8)


RUN: effb0_ft  (efficientnet_b0, freeze=False, epochs=8)
[train] run=effb0_ft device=cuda
[train] params total=4,015,234 trainable=4,015,234


[train] ep=00 tr_loss=1.3989 tr_acc=0.407 val_loss=1.1786 val_acc=0.521


[train] ep=01 tr_loss=1.2141 tr_acc=0.488 val_loss=1.0910 val_acc=0.562


[train] ep=02 tr_loss=1.1443 tr_acc=0.520 val_loss=1.0830 val_acc=0.539


[train] ep=03 tr_loss=1.0924 tr_acc=0.545 val_loss=1.0823 val_acc=0.585


[train] ep=04 tr_loss=1.0821 tr_acc=0.555 val_loss=1.0735 val_acc=0.568


[train] ep=05 tr_loss=1.0513 tr_acc=0.566 val_loss=1.0689 val_acc=0.594


[train] ep=06 tr_loss=1.0325 tr_acc=0.569 val_loss=1.0483 val_acc=0.601


[train] ep=07 tr_loss=1.0441 tr_acc=0.572 val_loss=1.0366 val_acc=0.600

[effb0_ft] DONE in 150.4 min
  accuracy             0.5895
  macro_f1             0.5493
  weighted_f1          0.5867
  macro_precision      0.5361
  macro_recall         0.5834
  quadratic_kappa      0.7465
  macro_auc            0.8932
  minutes              150.4136


{'accuracy': 0.5895054822107854,
 'macro_f1': 0.5493392989983227,
 'weighted_f1': 0.586668877384202,
 'macro_precision': 0.5360601087117595,
 'macro_recall': 0.583421318895923,
 'quadratic_kappa': 0.7465471945279865,
 'macro_auc': 0.8932002441375296,
 'run_name': 'effb0_ft',
 'best_epoch': 7,
 'minutes': 150.4135691245397}

In [21]:
!ls -la /content/drive/MyDrive/cdt-data/models/
!ls -la /content/drive/MyDrive/cdt-data/results/

total 1065106
-rw------- 1 root root  16361539 Apr 25 00:42 effb0_ft_best.pt
-rw------- 1 root root 537153365 Apr 24 11:57 vgg16_frozen_baseline_best.pt
-rw------- 1 root root 537152807 Apr 24 21:48 vgg16_ft_best.pt
total 781
-rw------- 1 root root    469 Apr 24 13:23 baselines.json
-rw------- 1 root root  88760 Apr 24 05:33 class_distribution.png
-rw------- 1 root root    785 Apr 25 00:46 finetuning_results.json
-rw------- 1 root root 383390 Apr 24 05:35 sample_clocks_by_class.png
-rw------- 1 root root 320681 Apr 24 05:54 smartcrop_before_after.png
drwx------ 3 root root   4096 Apr 24 07:34 vgg16_frozen_baseline


In [28]:
# we up the batch size for ViT (may be more memory-efficient than VGG16)
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_ds, batch_size=64, shuffle=True,
    num_workers=6, pin_memory=True, persistent_workers=True,
    prefetch_factor=4, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=64, shuffle=False,
    num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
test_loader = DataLoader(
    test_ds, batch_size=64, shuffle=False,
    num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
print(f"[ok] batch_size=64, train batches: {len(train_loader)}")

# we wuickly check for a batch of 30
import time, torch
t0 = time.time()
for i, (imgs, labels) in enumerate(train_loader):
    imgs = imgs.to('cuda', non_blocking=True)
    labels = labels.to('cuda', non_blocking=True)
    if i >= 29: break
elapsed = time.time() - t0
print(f"30 batches in {elapsed:.1f}s ({elapsed/30:.2f}s/batch)")
print(f"Projected ViT epoch time: {elapsed/30 * len(train_loader) / 60:.1f} min")

[ok] batch_size=64, train batches: 649
30 batches in 32.0s (1.07s/batch)
Projected ViT epoch time: 11.5 min


In [29]:
# ViT-B/16 fine-tuned
run_experiment('vit_b16', 'vit_b16_ft', epochs=8)


RUN: vit_b16_ft  (vit_b16, freeze=False, epochs=8)
[train] run=vit_b16_ft device=cuda
[train] params total=85,803,270 trainable=85,803,270


[train] ep=00 tr_loss=1.1081 tr_acc=0.519 val_loss=0.9401 val_acc=0.609


[train] ep=01 tr_loss=0.9360 tr_acc=0.605 val_loss=0.9024 val_acc=0.634


[train] ep=02 tr_loss=0.8673 tr_acc=0.636 val_loss=0.8924 val_acc=0.653


[train] ep=03 tr_loss=0.8032 tr_acc=0.653 val_loss=0.9078 val_acc=0.677


[train] ep=04 tr_loss=0.7758 tr_acc=0.665 val_loss=0.8918 val_acc=0.675


[train] ep=05 tr_loss=0.7249 tr_acc=0.679 val_loss=0.8705 val_acc=0.678


[train] ep=06 tr_loss=0.7022 tr_acc=0.683 val_loss=0.8958 val_acc=0.684


[train] ep=07 tr_loss=0.6890 tr_acc=0.685 val_loss=0.9069 val_acc=0.685

[vit_b16_ft] DONE in 116.1 min
  accuracy             0.6719
  macro_f1             0.6484
  weighted_f1          0.6714
  macro_precision      0.6328
  macro_recall         0.6739
  quadratic_kappa      0.8085
  macro_auc            0.9253
  minutes              116.0548


{'accuracy': 0.671850525844708,
 'macro_f1': 0.6483722009825265,
 'weighted_f1': 0.6714299063697348,
 'macro_precision': 0.6328339901983162,
 'macro_recall': 0.673890432218493,
 'quadratic_kappa': 0.808468356197879,
 'macro_auc': 0.9253265860583042,
 'run_name': 'vit_b16_ft',
 'best_epoch': 5,
 'minutes': 116.05480935176213}

In [30]:
import pandas as pd, json

# here we pull all of the metrics from disk
# this is resilient to runtime disconnects
results = {}
try:
    with open(cfg.RESULTS_DIR / 'baselines.json') as f:
        results.update(json.load(f))
except FileNotFoundError:
    print("[warning] baselines.json not found")

try:
    with open(cfg.RESULTS_DIR / 'finetuning_results.json') as f:
        results.update(json.load(f))
except FileNotFoundError:
    print("[warning] finetuning_results.json not found")

rows = []
metric_keys = ['accuracy', 'macro_f1', 'weighted_f1', 'macro_precision',
               'macro_recall', 'quadratic_kappa', 'macro_auc']
for run_name, m in results.items():
    if isinstance(m, dict):
        rows.append({'run': run_name, **{k: m.get(k) for k in metric_keys}})

df_results = pd.DataFrame(rows).set_index('run')
df_results = df_results.sort_values('quadratic_kappa', ascending=False)  # best first
df_results.to_csv(cfg.RESULTS_DIR / 'comparison.csv')

print("=" * 90)
print("THE FULL COMPARISON TABLE (sorted by quadratic_kappa)")
print("=" * 90)
print(df_results.round(4).to_string())
print(f"\nSaved to {cfg.RESULTS_DIR / 'comparison.csv'}")

THE FULL COMPARISON TABLE (sorted by quadratic_kappa)
                accuracy  macro_f1  weighted_f1  macro_precision  macro_recall  quadratic_kappa  macro_auc
run                                                                                                       
vit_b16_ft        0.6719    0.6484       0.6714           0.6328        0.6739           0.8085     0.9253
vgg16_ft          0.6160    0.5842       0.6120           0.5766        0.6043           0.7813     0.9026
effb0_ft          0.5895    0.5493       0.5867           0.5361        0.5834           0.7465     0.8932
vgg16_frozen      0.4542    0.4156       0.4487           0.4219        0.4251           0.6368     0.8001
majority_class    0.3698    0.0900       0.1997              NaN           NaN           0.0000        NaN

Saved to /content/drive/MyDrive/cdt-data/results/comparison.csv


In [31]:
!ls -la /content/drive/MyDrive/cdt-data/models/

total 1400336
-rw------- 1 root root  16361539 Apr 25 00:42 effb0_ft_best.pt
-rw------- 1 root root 537153365 Apr 24 11:57 vgg16_frozen_baseline_best.pt
-rw------- 1 root root 537152807 Apr 24 21:48 vgg16_ft_best.pt
-rw------- 1 root root 343275199 Apr 25 03:25 vit_b16_ft_best.pt
